# RGT-Est on your own SEG-Y: RGT, horizons and an RGT-guided LFM

Runs the pretrained RGT-Est model (Dou, Wu, Gao & Bi, 2026) on a SEG-Y exported from RokDoc, and writes the results back as files RokDoc can load:

| Output | File |
|---|---|
| RGT volume (same trace headers as your input) | `<input>_RGT.sgy` (and `<input>_RGT_guided.sgy` if you give horizons) |
| Horizons traced through chosen times at the well | `<input>_horizons.txt` (Inline, Crossline, X, Y, TWT) |
| Velocity model resampled onto the seismic grid (optional) | `<input>_Vint_on_grid.sgy` |
| Low-frequency model (optional, needs well logs): velocity background + the wells' detail along RGT, with a leave-one-well-out test | `<input>_LFM.sgy`, a decimated `<input>_LFM_stepN.sgy`, and `<input>_LFM_velocity_only.sgy` for comparison |

**Before you start**
1. **Runtime → Change runtime type → L4 GPU.**
2. Put the input files in your Google Drive folder (`DRIVE_FOLDER`, default **My Drive › PUBLIC**). Each run copies them into the session and saves the outputs back to **PUBLIC/outputs/<date_time>/**. Click **Connect to Google Drive** when asked.
   *(Or set `DATA_SOURCE = "upload"` and drag the files into the **Files** panel instead.)* The inputs:
   - `RGT-Est_CIG-Benchmark.pt` (the pretrained model)
   - your SEG-Y crop
   - *(optional, for horizon guidance)* RokDoc horizon exports (`.txt`: Inline, Crossline, TWT)
   - *(optional)* an interval-velocity SEG-Y in TWT: any extent or sampling, null padding is fine
   - *(optional, for the LFM)* well logs as **LAS files exported in TWT** (vertical wells; position read from `XWELL`/`YWELL` in the header). `.xlsx`/`.csv` with TWT and AI columns also work, with the position set in `WELL_LOCATIONS`.
3. Check the **Settings** cell, then **Runtime → Run all**.

**Keeping your data private:** this notebook never uploads your data anywhere. But if you use *File → Save a copy in GitHub*, **untick "Include cell outputs"**, or the plots and printed values will be published with the notebook.

## 1. Settings

In [ ]:
DATA_SOURCE   = "drive"                           # "drive" = inputs from Google Drive, outputs saved there
                                                  # "upload" = files dragged into the Files panel
DRIVE_FOLDER  = "/content/drive/MyDrive/PUBLIC"   # your Drive folder (My Drive › PUBLIC)
DATA_FOLDER   = "/content"                        # working folder inside the session
MODEL_FILE    = "RGT-Est_CIG-Benchmark.pt"
SEGY_FILE     = "Full_Stack_8_41_CROP_1793889427461.sgy"

# SEG-Y header byte positions for inline / crossline. None = detect automatically.
IL_BYTE, XL_BYTE = None, None

INTERVAL_MS   = (1300, 2300)        # interval of interest (for plots)
WELL_IL, WELL_XL = 1123, 2098       # well location

# Horizon guidance (optional): RokDoc horizon .txt exports in TWT. Empty list = unguided only.
HORIZON_FILES = ["Jubilee.txt", "Ranger.txt"]   # [] = unguided only
USE_GUIDED    = True                # use the guided RGT for the horizon/LFM outputs below

# Horizons: traced through these TWTs at the well, then followed across the survey by the RGT
HORIZON_TIMES_AT_WELL = [1300, 1550, 1800, 2050, 2300]

# Low-frequency model from wells (skipped if there are none)
WELL_FILES    = "auto"              # "auto" = every .las file in the data folder, or a list, e.g. ["NORTON_1.las", ...]
WELL_AI_CURVE = "AI_WTR"            # impedance curve in the LAS files (AI_WTR is usually brine-substituted)
WELL_LOCATIONS = {}                 # optional overrides {"NORTON_1": (inline, crossline)}; default: XWELL/YWELL in the LAS
IDW_POWER     = 2                   # how fast a well's influence fades with distance (inverse-distance weighting)
LFM_CUTOFF_HZ = 8                   # low-pass cut-off for the log
LFM_TRACE_STEP = 8                  # also write a decimated LFM, every Nth inline and crossline
LFM_TAPER_MS  = 100                 # above/below each log, fade from the well's detail to the background

# Velocity model (optional): interval velocity SEG-Y in TWT. Its extent and sampling can differ from
# the seismic; null-padded traces (zeros, -999.25, ...) are dropped and gaps are interpolated.
VELOCITY_FILE = "Vint_Vint_1793927905561.sgy" # "" = no velocity model
VEL_IL_BYTE, VEL_XL_BYTE = None, None   # None = detect automatically
VEL_MATCH     = "auto"              # "auto", "ilxl" (by inline/crossline headers) or "xy" (by X/Y coordinates)

## 2. Set up: GPU, libraries, files

In [ ]:
!pip -q install segyio lasio
import os, glob, torch
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("No GPU found. Go to Runtime → Change runtime type → GPU, then run again.")

import time
RUN_START = time.time()

import shutil

def session_files():
    # Every file in the session, without descending into Google Drive or Colab's sample data
    for root, dirs, fnames in os.walk(DATA_FOLDER):
        dirs[:] = [d for d in dirs if d not in ("drive", "sample_data") and not d.startswith(".")]
        for f in fnames:
            yield os.path.join(root, f)

def find_file(name):
    # Exact name only (in DATA_FOLDER or any subfolder). Never substitutes another file.
    if not name:
        return None
    p = os.path.join(DATA_FOLDER, name)
    if os.path.exists(p):
        return p
    hits = [h for h in session_files() if os.path.basename(h) == name]
    return hits[0] if hits else None

def list_uploaded(*exts):
    return sorted(os.path.relpath(h, DATA_FOLDER) for h in session_files() if h.endswith(exts))

# ---- copy the inputs in from Google Drive
WELL_LIST = [] if WELL_FILES == "auto" else list(WELL_FILES)
INPUTS = [f for f in [MODEL_FILE, SEGY_FILE, VELOCITY_FILE] + list(HORIZON_FILES) + WELL_LIST if f]
if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_FOLDER):
        raise FileNotFoundError(f"Drive folder not found: {DRIVE_FOLDER}. Check DRIVE_FOLDER in Settings.")
    if WELL_FILES == "auto":                                   # every .las in the Drive folder
        INPUTS += sorted(f for f in os.listdir(DRIVE_FOLDER) if f.lower().endswith(".las"))
    for f in INPUTS:
        src, dst = os.path.join(DRIVE_FOLDER, f), os.path.join(DATA_FOLDER, f)
        if not os.path.exists(src):
            print(f"not in Drive   {f}")
        elif os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            print(f"already here   {f}")
        else:
            t0 = time.time(); shutil.copyfile(src, dst)
            print(f"copied         {f}  ({os.path.getsize(dst)/1e6:.0f} MB, {time.time()-t0:.0f} s)")
    print()

MODEL_PATH = find_file(MODEL_FILE)
SEGY_PATH  = find_file(SEGY_FILE)
VEL_PATH   = find_file(VELOCITY_FILE) if VELOCITY_FILE else None
for label, p, needed in (("model", MODEL_PATH, True), ("SEG-Y", SEGY_PATH, True),
                         ("velocity", VEL_PATH, bool(VELOCITY_FILE))):
    if p:        print(f"found    {label:9s} {p}  ({os.path.getsize(p)/1e6:.0f} MB)")
    elif needed: print(f"MISSING  {label}")
    else:        print(f"(no {label} file set: optional, skipped)")
HORIZON_PATHS = []
for h in HORIZON_FILES:
    p = find_file(h)
    print(f"found    horizon   {p}" if p else f"MISSING  horizon {h}")
    if not p:
        txts = list_uploaded(".txt")
        raise FileNotFoundError(f"Horizon file {h!r} not found. .txt files uploaded: {txts}")
    HORIZON_PATHS.append(p)
if WELL_FILES == "auto":
    WELL_PATHS = sorted(h for h in session_files() if h.lower().endswith(".las"))
else:
    WELL_PATHS = [find_file(f) for f in WELL_LIST]
    for f, p in zip(WELL_LIST, WELL_PATHS):
        if not p: print(f"MISSING  well     {f}")
for p in WELL_PATHS:
    if p: print(f"found    well      {p}")
if not WELL_PATHS:
    print("(no well logs found: the LFM will be skipped)")
missing_wells = None in WELL_PATHS
if not (MODEL_PATH and SEGY_PATH) or (VELOCITY_FILE and not VEL_PATH) or missing_wells:
    print("\nFiles currently in the session:")
    print("  SEG-Y:", list_uploaded(".sgy", ".segy", ".SGY", ".SEGY") or "none")
    print("  model:", list_uploaded(".pt") or "none")
    print("  other:", list_uploaded(".txt", ".las", ".LAS", ".xlsx", ".csv") or "none")
    raise FileNotFoundError("Upload the missing file(s) in the Files panel, or fix the names in Settings, then run this cell again. "
                            "(Files ending _RGT / _LFM / _Vint_on_grid are outputs from earlier runs, not inputs.)")
for label, p in (("SEG-Y", SEGY_PATH), ("velocity", VEL_PATH)):
    if p and any(tag in os.path.basename(p) for tag in ("_RGT", "_LFM", "_Vint_on_grid")):
        raise ValueError(f"The {label} file {os.path.basename(p)} looks like an OUTPUT of an earlier run, not an input. "
                         "Set the original file name in Settings.")

!git clone -q https://github.com/fergusrssmith/RGT /content/RGT 2>/dev/null || git -C /content/RGT pull -q
import sys
sys.path.insert(0, "/content/RGT/demo")
sys.path.insert(0, "/content/RGT/colab")
import importlib, rgt_lfm as R
importlib.reload(R)

## 3. Read the SEG-Y and check the geometry

**Compare the printout with RokDoc's 3D Geometry dialog** (inline/crossline ranges, sample rate, corner coordinates). If they match, the output files will land in the right place.

In [ ]:
import numpy as np
import segyio

G = R.read_segy_cube(SEGY_PATH, IL_BYTE, XL_BYTE)
cube, ilines, xlines, samples, dt = G["cube"], G["ilines"], G["xlines"], G["samples"], G["dt"]
il_idx, xl_idx, trace_index, X, Y = G["il_idx"], G["xl_idx"], G["trace_index"], G["X"], G["Y"]
n_tr, il_byte, xl_byte = G["n_tr"], G["il_byte"], G["xl_byte"]
NI, NX, NT = cube.shape

print(f"Header bytes used:  inline {il_byte}, crossline {xl_byte}")
print(f"Inlines:    {ilines[0]} – {ilines[-1]}  ({NI})")
print(f"Crosslines: {xlines[0]} – {xlines[-1]}  ({NX})")
print(f"TWT:        {samples[0]:.0f} – {samples[-1]:.0f} ms, every {dt:g} ms  ({NT} samples)")
print(f"Traces:     {n_tr}  (missing grid nodes: {(trace_index < 0).sum()})")
print("Corner coordinates (inline, crossline → X, Y):")
for i, j in ((0, 0), (0, NX - 1), (NI - 1, 0), (NI - 1, NX - 1)):
    print(f"   {ilines[i]:>6}, {xlines[j]:>6} → {X[i, j]:,.1f}, {Y[i, j]:,.1f}")
print(f"Amplitudes: min {cube.min():.3g}, max {cube.max():.3g}, std {cube.std():.3g}")

if WELL_IL in ilines and WELL_XL in xlines:
    wi, wx = np.searchsorted(ilines, WELL_IL), np.searchsorted(xlines, WELL_XL)
    print(f"Well at inline {WELL_IL}, crossline {WELL_XL} → X {X[wi, wx]:,.1f}, Y {Y[wi, wx]:,.1f}")
else:
    wi = wx = None
    print(f"WARNING: the well (inline {WELL_IL}, crossline {WELL_XL}) is outside this volume.")

In [ ]:
import matplotlib.pyplot as plt

def tidx(ms):
    return int(np.clip(round((ms - samples[0]) / dt), 0, NT - 1))

ci = wi if wi is not None else NI // 2
cx_ = wx if wx is not None else NX // 2
v = np.percentile(np.abs(cube), 99)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-v, vmax=v,
               extent=(xlines[0], xlines[-1], samples[-1], samples[0]))
axes[0].set_title(f"inline {ilines[ci]}"); axes[0].set_xlabel("crossline")
axes[1].imshow(cube[:, cx_].T, cmap="gray", aspect="auto", vmin=-v, vmax=v,
               extent=(ilines[0], ilines[-1], samples[-1], samples[0]))
axes[1].set_title(f"crossline {xlines[cx_]}"); axes[1].set_xlabel("inline")
for ax in axes[:2]:
    ax.set_ylabel("TWT (ms)")
    for t in INTERVAL_MS: ax.axhline(t, color="orange", lw=1, ls="--")
axes[0].axvline(xlines[cx_], color="red", lw=1, ls=":"); axes[1].axvline(ilines[ci], color="red", lw=1, ls=":")
tm = sum(INTERVAL_MS) / 2
axes[2].imshow(cube[:, :, tidx(tm)], cmap="gray", aspect="auto", vmin=-v, vmax=v, origin="lower",
               extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
axes[2].plot(WELL_XL, WELL_IL, "r*", ms=14)
axes[2].set_title(f"time slice {tm:.0f} ms (red star = well)"); axes[2].set_xlabel("crossline"); axes[2].set_ylabel("inline")
fig.suptitle("QC: input seismic (orange dashes = interval of interest)"); fig.tight_layout(); plt.show()

## 3b. Load the guidance horizons (skipped if `HORIZON_FILES` is empty)

Points are matched to the crop by their inline/crossline numbers. Anything outside the crop (or outside its TWT window), and null values, are dropped.

In [ ]:
import re

HZ = []
for path in HORIZON_PATHS:
    name, ztype, rows = os.path.splitext(os.path.basename(path))[0], None, []
    with open(path, errors="ignore") as fh:
        for line in fh:
            m = re.match(r'\s*Horizon\s+"(.+?)"', line)
            if m: name = m.group(1)
            if "Z Type" in line: ztype = line.split(":")[-1].strip()
            parts = line.replace(",", " ").split()
            if len(parts) >= 3:
                try: rows.append([float(v) for v in parts[:3]])
                except ValueError: pass
    a = np.array(rows, dtype=float)
    if ztype and ztype.upper() != "TWT":
        print(f"WARNING: {name} has Z type {ztype!r}, but the seismic is in TWT.")
    hil, hxl, hz = np.round(a[:, 0]), np.round(a[:, 1]), a[:, 2]
    ok = np.isfinite(hz) & (hz > -999) & np.isin(hil, ilines) & np.isin(hxl, xlines) \
         & (hz >= samples[0]) & (hz <= samples[-1])
    t = np.full((NI, NX), np.nan)
    t[np.searchsorted(ilines, hil[ok]), np.searchsorted(xlines, hxl[ok])] = hz[ok]
    if not np.isfinite(t).any():
        print(f"{name}: no points fall on the crop (inline/crossline/TWT): skipped."); continue
    HZ.append(dict(name=name, t=t))
    print(f"{name}: {len(a):,} points in file, {np.isfinite(t).sum():,} on the crop "
          f"({100*np.isfinite(t).mean():.0f}% of traces), TWT {np.nanmin(t):.0f} – {np.nanmax(t):.0f} ms")

HZ.sort(key=lambda h: np.nanmean(h["t"]))                       # shallow → deep
for up, down in zip(HZ, HZ[1:]):
    both = np.isfinite(up["t"]) & np.isfinite(down["t"])
    bad = (down["t"][both] <= up["t"][both]).mean() * 100
    gap = np.median(down["t"][both] - up["t"][both])
    print(f"{up['name']} → {down['name']}: median gap {gap:.0f} ms, crossing/touching on {bad:.1f}% of traces"
          + ("   ← WARNING: horizons should not cross" if bad > 0.5 else ""))

if HZ:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
    v = np.percentile(np.abs(cube), 99)
    for ax, sec, ext, xl_, hsl in (
            (axes[0], cube[ci].T, (xlines[0], xlines[-1], samples[-1], samples[0]), f"inline {ilines[ci]}", lambda t: (xlines, t[ci])),
            (axes[1], cube[:, cx_].T, (ilines[0], ilines[-1], samples[-1], samples[0]), f"crossline {xlines[cx_]}", lambda t: (ilines, t[:, cx_]))):
        ax.imshow(sec, cmap="gray", aspect="auto", vmin=-v, vmax=v, extent=ext)
        for k, h in enumerate(HZ):
            ax.plot(*hsl(h["t"]), lw=1.5, color=f"C{k+1}", label=h["name"])
        ax.set_title(f"{xl_}: guidance horizons"); ax.set_ylabel("TWT (ms)"); ax.legend()
    fig.tight_layout(); plt.show()

## 3c. Load the velocity model (skipped if `VELOCITY_FILE` is empty)

Velocity traces are placed by their inline/crossline headers (or X/Y), null-padded traces are dropped, partly-null traces are repaired, and the result is resampled onto every seismic trace and sample.
Gaps between velocity traces are interpolated; seismic traces beyond the last valid velocity trace take the nearest one.

In [ ]:
VEL = None
if VEL_PATH:
    from matplotlib.colors import ListedColormap
    VEL, vel_info = R.load_velocity_on_grid(VEL_PATH, G, VEL_IL_BYTE, VEL_XL_BYTE, match=VEL_MATCH)
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), gridspec_kw={"width_ratios": [1.2, 2, 2]})
    axes[0].imshow(vel_info["source"], origin="lower", aspect="auto", vmin=-0.5, vmax=2.5,
                   cmap=ListedColormap(["#2a9d8f", "#e9c46a", "#e76f51"]),
                   extent=(xlines[0] - .5, xlines[-1] + .5, ilines[0] - .5, ilines[-1] + .5))
    axes[0].plot(WELL_XL, WELL_IL, "r*", ms=14)
    axes[0].set_title("velocity source per trace\ngreen = exact, yellow = interpolated, red = nearest")
    axes[0].set_xlabel("crossline"); axes[0].set_ylabel("inline")
    for ax, vsec, ssec, ext, title, mark in (
            (axes[1], VEL[ci].T, cube[ci].T, (xlines[0], xlines[-1], samples[-1], samples[0]), f"Vint inline {ilines[ci]}", xlines[cx_]),
            (axes[2], VEL[:, cx_].T, cube[:, cx_].T, (ilines[0], ilines[-1], samples[-1], samples[0]), f"Vint crossline {xlines[cx_]}", ilines[ci])):
        im = ax.imshow(vsec, cmap="turbo", aspect="auto", extent=ext)
        v = np.percentile(np.abs(ssec), 99)
        ax.imshow(ssec, cmap="gray", aspect="auto", extent=ext, vmin=-v, vmax=v, alpha=0.3)
        ax.axvline(mark, color="w", lw=1, ls=":"); ax.set_title(title); ax.set_ylabel("TWT (ms)")
        plt.colorbar(im, ax=ax, label="Vint (m/s)")
    axes[1].set_xlabel("crossline"); axes[2].set_xlabel("inline")
    fig.tight_layout(); plt.show()
else:
    print("No VELOCITY_FILE set: skipping.")

## 4. Run RGT-Est

The model works on a fixed 400 × 512 × 512 grid. For your crop (256 inlines at 25 m, 512 crosslines at 12.5 m), the inline direction is stretched ×2, which makes the grid 12.5 m in both directions. The result is then resized back to your exact grid.

In [ ]:
import time
import torch.nn as nn
import torch.nn.functional as F

def normalization(data):
    rng = np.max(data) - np.min(data)
    return (data - np.min(data)) / (rng + 1e-6)

def z_score_clip(data, clp_s=2.0):
    z = (data - np.mean(data)) / (np.std(data) + 1e-8)
    return normalization(np.clip(z, -clp_s, clp_s))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.jit.load(MODEL_PATH, map_location=device).eval()

from scipy.ndimage import grey_dilation
GRID = (400, 512, 512)
seis_norm = z_score_clip(cube) * 2 - 1

def guidance_volume(use):
    """Horizons → the model's guidance channel (same recipe as the authors' F3 demo)."""
    ids = np.zeros((NI, NX, NT), np.int16)
    for k, h in enumerate(use, 1):
        i, j = np.nonzero(np.isfinite(h["t"]))
        ids[i, j, np.clip(np.round((h["t"][i, j] - samples[0]) / dt).astype(int), 0, NT - 1)] = k
    ids = grey_dilation(ids, size=2)
    vol = np.zeros((NI, NX, NT), np.float32)
    for k, h in enumerate(use, 1):
        vol[ids == k] = (np.nanmean(h["t"]) - samples[0]) / dt / NT * 2 - 1
    return vol

def run_rgt(cube, guide=None):
    s = F.interpolate(torch.from_numpy(seis_norm.transpose(2, 0, 1).copy())[None, None], GRID, mode="trilinear")
    if guide is None:
        g = torch.zeros_like(s)
    else:
        g = F.interpolate(torch.from_numpy(guide.transpose(2, 0, 1).copy())[None, None], GRID, mode="nearest")
    x = torch.cat([s, g, (g != 0).float()], dim=1).to(device)
    t0 = time.time()
    with torch.no_grad(), torch.autocast(device_type=device):
        rgt = model(nn.ReplicationPad3d(8)(x))[:, :, 8:-8, 8:-8, 8:-8]
    print(f"inference took {time.time()-t0:.1f} s")
    out = F.interpolate(rgt.float(), (NT, NI, NX), mode="trilinear").cpu().numpy()[0, 0]
    del x, rgt
    if device == "cuda": torch.cuda.empty_cache()
    return z_score_clip(out).transpose(1, 2, 0).astype(np.float32)        # (inline, crossline, time), 0 = top

# ---- runs ----
print("Unguided:"); rgt_unguided = run_rgt(cube)
rgt_guided, rgt_loo = None, {}
if HZ:
    print(f"Guided with all {len(HZ)} horizons:"); rgt_guided = run_rgt(cube, guidance_volume(HZ))
    if len(HZ) >= 2:                                   # leave-one-out runs, for a fair score
        for h in HZ:
            print(f"Guided without {h['name']} (to test on it):")
            rgt_loo[h["name"]] = run_rgt(cube, guidance_volume([o for o in HZ if o is not h]))
rgt = rgt_guided if (rgt_guided is not None and USE_GUIDED) else rgt_unguided
print("Using the", "GUIDED" if rgt is rgt_guided else "UNGUIDED", "RGT for the outputs below. Shape:", rgt.shape)

## 4b. Does guidance help? (needs horizons)

For each horizon, we take the single RGT value that best matches it, trace the predicted horizon, and measure the vertical miss (ms) on every trace.

- **unguided:** the model saw no horizons.
- **held out:** the model was guided by the *other* horizon(s) but not this one. **This is the fair test.**
- **all guided:** the model was given this horizon too, so this only shows how closely it honours its input.

Hit rate = % of traces within 2 samples.

In [ ]:
import pandas as pd
from rgt_helper import horizons_from_rgt

def horizon_errors(vol, hz_list):
    levels = []
    for h in hz_list:
        i, j = np.nonzero(np.isfinite(h["t"]))
        ti = np.clip(np.round((h["t"][i, j] - samples[0]) / dt).astype(int), 0, NT - 1)
        levels.append(np.median(vol[i, j, ti]))
    pred = samples[0] + horizons_from_rgt(sig=0.0, hu=np.array(levels), ux=vol) * dt
    out = []
    for h, p in zip(hz_list, pred):
        m = np.isfinite(h["t"]); err = np.abs(p[m] - h["t"][m])
        out.append((err.mean(), 100 * (err <= 2 * dt).mean()))
    return out

if not HZ:
    print("No horizons given: nothing to compare.")
else:
    res = {"unguided": horizon_errors(rgt_unguided, HZ), "all guided": horizon_errors(rgt_guided, HZ)}
    if rgt_loo:
        res["held out"] = [horizon_errors(rgt_loo[h["name"]], [h])[0] for h in HZ]
    cols = [c for c in ("unguided", "held out", "all guided") if c in res]
    table = pd.DataFrame({"horizon": [h["name"] for h in HZ]})
    for c in cols:
        table[f"{c}: error (ms)"] = [e for e, _ in res[c]]
    for c in cols:
        table[f"{c}: hit %"] = [r for _, r in res[c]]
    pd.options.display.float_format = "{:.1f}".format
    display(table)

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    v = np.percentile(np.abs(cube), 99)
    for row, (sec_s, get, ext, title) in enumerate((
            (cube[ci].T, lambda vol: vol[ci].T, (xlines[0], xlines[-1], samples[-1], samples[0]), f"inline {ilines[ci]}"),
            (cube[:, cx_].T, lambda vol: vol[:, cx_].T, (ilines[0], ilines[-1], samples[-1], samples[0]), f"crossline {xlines[cx_]}"))):
        xs = xlines if row == 0 else ilines
        for col, (name, vol) in enumerate((("unguided", rgt_unguided), ("guided (all horizons)", rgt_guided))):
            ax = axes[row, col]
            ax.imshow(sec_s, cmap="gray", aspect="auto", vmin=-v, vmax=v, extent=ext)
            ax.contour(get(vol), levels=np.linspace(0.02, 0.98, 45), cmap="jet", linewidths=0.9,
                       extent=(ext[0], ext[1], ext[3], ext[2]))
            for h in HZ:
                ax.plot(xs, h["t"][ci] if row == 0 else h["t"][:, cx_], color="magenta", lw=2.5, ls="--")
            ax.set_title(f"{title}: {name}  (magenta dashes = interpreted horizons)"); ax.set_ylabel("TWT (ms)")
            ax.set_ylim(samples[-1], samples[0])
    fig.tight_layout(); plt.show()

## 5. Look at the result through the well

Seismic, RGT, and seismic with horizons (RGT contours). Check that the lines follow the reflectors, especially between the orange dashes.

In [ ]:
levels = np.linspace(0.02, 0.98, 45)

def three_panel(sec_s, sec_r, extent, title, xlabel, mark=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    v = np.percentile(np.abs(sec_s), 99)
    axes[0].imshow(sec_s, cmap="gray", aspect="auto", vmin=-v, vmax=v, extent=extent)
    axes[1].imshow(sec_r, cmap="jet", aspect="auto", extent=extent)
    axes[2].imshow(sec_s, cmap="gray", aspect="auto", vmin=-v, vmax=v, extent=extent)
    axes[2].contour(sec_r, levels=levels, cmap="jet", linewidths=1.0, extent=(extent[0], extent[1], extent[3], extent[2]))
    for ax, name in zip(axes, ("seismic", "RGT", "seismic + horizons")):
        ax.set_title(f"{title}: {name}"); ax.set_xlabel(xlabel); ax.set_ylabel("TWT (ms)")
        for t in INTERVAL_MS: ax.axhline(t, color="orange", lw=1, ls="--")
        if mark is not None: ax.axvline(mark, color="red", lw=1, ls=":")
        ax.set_ylim(samples[-1], samples[0])
    fig.tight_layout(); plt.show()

three_panel(cube[ci].T, rgt[ci].T, (xlines[0], xlines[-1], samples[-1], samples[0]),
            f"inline {ilines[ci]}", "crossline", WELL_XL if wx is not None else None)
three_panel(cube[:, cx_].T, rgt[:, cx_].T, (ilines[0], ilines[-1], samples[-1], samples[0]),
            f"crossline {xlines[cx_]}", "inline", WELL_IL if wi is not None else None)
for k in (NI // 4, 3 * NI // 4):
    three_panel(cube[k].T, rgt[k].T, (xlines[0], xlines[-1], samples[-1], samples[0]), f"inline {ilines[k]}", "crossline")

## 6. Save the RGT as SEG-Y (all trace headers copied from your input)

In [ ]:
import shutil
stem = os.path.splitext(os.path.basename(SEGY_PATH))[0]
OUT = {}

def write_like_input(volume, out_path):
    """Copy the input SEG-Y, then replace every trace with the matching column of `volume`."""
    shutil.copyfile(SEGY_PATH, out_path)
    with segyio.open(out_path, "r+", ignore_geometry=True) as f:
        for k in range(n_tr):
            f.trace[k] = np.ascontiguousarray(volume[il_idx[k], xl_idx[k]], dtype=np.float32)
    print("wrote", out_path, f"({os.path.getsize(out_path)/1e6:.0f} MB)")
    return out_path

OUT["rgt"] = write_like_input(rgt_unguided, f"/content/{stem}_RGT.sgy")
if rgt_guided is not None:
    OUT["rgt_guided"] = write_like_input(rgt_guided, f"/content/{stem}_RGT_guided.sgy")
print("RGT values run from 0 (top of volume, youngest) to 1 (base, oldest).")
if VEL is not None:
    OUT["vint"] = write_like_input(VEL, f"/content/{stem}_Vint_on_grid.sgy")

## 7. Trace horizons and export them for RokDoc

Each horizon goes through one of `HORIZON_TIMES_AT_WELL` at the well, then follows the RGT across the volume.
Saved as ASCII columns: **Inline, Crossline, X, Y, TWT (ms)**.

In [ ]:
from rgt_helper import horizons_from_rgt
import pandas as pd

ref_i = wi if wi is not None else NI // 2
ref_x = wx if wx is not None else NX // 2
well_rgt = np.maximum.accumulate(rgt[ref_i, ref_x])                      # RGT down the well trace
levels_h = np.interp(HORIZON_TIMES_AT_WELL, samples, well_rgt)

hz_idx = horizons_from_rgt(sig=1.0, hu=levels_h, ux=rgt)                 # (n_horizons, inline, crossline), in samples
hz_ms = samples[0] + hz_idx * dt

II, XX = np.meshgrid(ilines, xlines, indexing="ij")
rows = []
for t_well, hz in zip(HORIZON_TIMES_AT_WELL, hz_ms):
    print(f"horizon {t_well} ms: at the well {hz[ref_i, ref_x]:.0f} ms, range across the volume {hz.min():.0f} – {hz.max():.0f} ms")
    rows.append(pd.DataFrame({"Horizon": f"RGT_{t_well}ms", "Inline": II.ravel(), "Crossline": XX.ravel(),
                              "X": X.ravel(), "Y": Y.ravel(), "TWT_ms": hz.ravel()}))
hz_table = pd.concat(rows)

OUT["horizons"] = f"/content/{stem}_horizons.txt"
hz_table.to_csv(OUT["horizons"], sep="\t", index=False, float_format="%.2f")
for t_well, g in hz_table.groupby("Horizon", sort=False):                 # also one file per horizon
    g.drop(columns="Horizon").to_csv(f"/content/{stem}_{t_well}.txt", sep="\t", index=False, float_format="%.2f")
print("wrote", OUT["horizons"], "and one file per horizon")

fig, axes = plt.subplots(1, len(hz_ms), figsize=(4.2 * len(hz_ms), 4))
for ax, hz, t_well in zip(np.atleast_1d(axes), hz_ms, HORIZON_TIMES_AT_WELL):
    im = ax.imshow(hz, cmap="viridis_r", aspect="auto", origin="lower",
                   extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
    ax.plot(xlines[ref_x], ilines[ref_i], "r*", ms=12)
    ax.set_title(f"horizon through {t_well} ms"); ax.set_xlabel("crossline"); ax.set_ylabel("inline")
    plt.colorbar(im, ax=ax, label="TWT (ms)")
fig.tight_layout(); plt.show()

## 8. (Optional) Low-frequency model from the wells

For each well (LAS in TWT): the log is low-pass filtered (`LFM_CUTOFF_HZ`) on its own sampling, then resampled to the seismic.

1. **Background.** With a velocity model: `AI ≈ a · Vint^b`, fitted on all wells together (no density assumption). Without one: an AI-vs-TWT trend from the wells.
2. **Well detail along RGT.** Each well's residual (log minus background) is placed on the RGT axis. Every point in the cube takes an inverse-distance-weighted blend of the wells' residuals **at its own RGT value**, so each layer is interpolated along itself. Above and below a log, that well's influence fades out over `LFM_TAPER_MS`; where no well has data, the background takes over.
3. **Low-pass** the sum.

**Leave-one-well-out test** (2+ wells): each well is predicted from the others and compared with its own low-passed log, for three methods. The background alone; wells interpolated along **flat time** (what you'd get without RGT); and wells interpolated along **RGT**. The gap between the last two is what RGT adds.

In [ ]:
from scipy.signal import butter, sosfiltfilt
import lasio

def _hdr(las, *keys):
    for k in keys:
        try:
            v = las.well[k].value
            if v not in (None, ""):
                return v
        except (KeyError, AttributeError):
            pass
    return None

def read_well(path):
    # Returns name, TWT (ms), AI, (X, Y) or None. LAS must be indexed in TWT; xlsx/csv = TWT, AI columns.
    base = os.path.splitext(os.path.basename(path))[0]
    if path.lower().endswith(".las"):
        las = lasio.read(path)
        name = str(_hdr(las, "WELL") or base).strip()
        unit, zstk = (las.curves[0].unit or "").lower(), str(_hdr(las, "ZSTK") or "").upper()
        if "ms" not in unit and zstk != "TWT":
            raise ValueError(f"{base}: the LAS index is in {unit!r} (Z-stick {zstk!r}), not TWT. Export it in TWT (ms).")
        if WELL_AI_CURVE not in las.keys():
            raise ValueError(f"{name}: no curve {WELL_AI_CURVE!r}. Curves in the file: {list(las.keys())}")
        t, ai = np.asarray(las.index, float), np.asarray(las[WELL_AI_CURVE], float)
        xy = None
        for kx, ky in (("XWELL", "YWELL"), ("X", "Y"), ("XCOORD", "YCOORD")):
            try:
                xy = (float(_hdr(las, kx)), float(_hdr(las, ky))); break
            except (TypeError, ValueError):
                pass
    else:
        raw = pd.read_excel(path, header=None) if path.lower().endswith((".xlsx", ".xls")) \
              else pd.read_csv(path, header=None, sep=None, engine="python")
        num = raw.apply(pd.to_numeric, errors="coerce").iloc[:, :2].dropna()
        t, ai, name, xy = num.iloc[:, 0].to_numpy(float), num.iloc[:, 1].to_numpy(float), base, None
    ok = np.isfinite(t) & np.isfinite(ai) & (ai > 0)                  # drops nulls (-999.25 etc.)
    o = np.argsort(t[ok])
    return name, t[ok][o], ai[ok][o], xy

to_xy, to_ilxl = R.grid_transforms(G)
II, XX = np.meshgrid(ilines, xlines, indexing="ij")
Xg = np.where(np.isfinite(X), X, R._apply(to_xy[0], II, XX))           # X/Y of every grid node
Yg = np.where(np.isfinite(Y), Y, R._apply(to_xy[1], II, XX))
t_norm = (samples - samples[0]) / (samples[-1] - samples[0])

wells = []
for path in WELL_PATHS:
    name, t, a_raw, xy = read_well(path)
    if name in WELL_LOCATIONS:
        il_w, xl_w = map(float, WELL_LOCATIONS[name]); src = "Settings"
    elif xy is not None:
        il_w, xl_w = R._apply(to_ilxl[0], *xy), R._apply(to_ilxl[1], *xy); src = "LAS X/Y"
    else:
        print(f"{name}: no X/Y in the file. Add it to WELL_LOCATIONS in Settings. Skipped."); continue
    if not (ilines[0] - 0.5 <= il_w <= ilines[-1] + 0.5 and xlines[0] - 0.5 <= xl_w <= xlines[-1] + 0.5):
        print(f"{name}: inline {il_w:.1f}, crossline {xl_w:.1f} is outside this volume. Skipped."); continue
    if len(t) < 10 or t.max() < samples[0] or t.min() > samples[-1]:
        print(f"{name}: the log (TWT {t.min():.0f}–{t.max():.0f} ms) doesn't overlap the seismic. Skipped."); continue
    # low-pass on the log's own (regular) sampling first, then resample to the seismic: avoids aliasing
    dtl = float(np.median(np.diff(t)))
    tt = np.arange(t[0], t[-1] + dtl / 2, dtl)
    aa = np.interp(tt, t, a_raw)
    low = sosfiltfilt(butter(4, LFM_CUTOFF_HZ, fs=1000.0 / dtl, output="sos"), aa) if len(tt) > 30 else aa
    ai_s = np.interp(samples, tt, aa, left=np.nan, right=np.nan)
    ai_low = np.interp(samples, tt, low, left=np.nan, right=np.nan)
    valid = np.isfinite(ai_low)
    if valid.sum() < 10:
        print(f"{name}: fewer than 10 samples inside the seismic window. Skipped."); continue
    i_w, j_w = int(np.argmin(np.abs(ilines - il_w))), int(np.argmin(np.abs(xlines - xl_w)))
    x_w, y_w = xy if xy is not None else (Xg[i_w, j_w], Yg[i_w, j_w])
    wells.append(dict(name=name, il=il_w, xl=xl_w, i=i_w, j=j_w, x=x_w, y=y_w, src=src, ai=ai_s, ai_low=ai_low,
                      valid=valid, first=int(np.argmax(valid)), last=int(NT - 1 - np.argmax(valid[::-1]))))
    print(f"{name:12s} inline {il_w:7.1f}  crossline {xl_w:7.1f}  ({src})   log TWT {t.min():.0f}–{t.max():.0f} ms every {dtl:g} ms"
          f"   AI {np.nanmin(ai_s):.0f}–{np.nanmax(ai_s):.0f}")

if not wells:
    print("No usable wells: skipping the LFM.")
else:
    sos = butter(4, LFM_CUTOFF_HZ, fs=1000.0 / dt, output="sos")
    lowpass = lambda v: sosfiltfilt(sos, v, axis=-1).astype(np.float32)
    K = 1001; U = np.linspace(0, 1, K)

    def fit_background(ws):
        y = np.concatenate([w["ai_low"][w["valid"]] for w in ws])
        if VEL is not None:                                    # AI ≈ a · Vint^b, all wells together
            v = np.concatenate([VEL[w["i"], w["j"]][w["valid"]] for w in ws]).astype(float)
            b, ln_a = np.polyfit(np.log(v), np.log(y), 1)
            if not 0.3 < b < 3:
                b, ln_a = 1.0, np.mean(np.log(y) - np.log(v))
            return ("velocity", np.exp(ln_a), b)
        tw = np.concatenate([samples[w["valid"]] for w in ws])  # no velocity: AI trend with TWT
        c1, c0 = np.polyfit(tw, y, 1)
        return ("trend", c0, c1)

    def bg_trace(m, i, j):
        return m[1] * VEL[i, j].astype(float) ** m[2] if m[0] == "velocity" else m[1] + m[2] * samples

    def bg_volume(m):
        if m[0] == "velocity":
            return m[1] * VEL.astype(np.float32) ** np.float32(m[2])
        return np.broadcast_to((m[1] + m[2] * samples).astype(np.float32), (NI, NX, NT)).copy()

    def well_profile(w, coord):
        if coord == "time":
            return t_norm
        return np.maximum.accumulate(rgt[w["i"], w["j"]]) + np.arange(NT) * 1e-7

    def well_table(w, m, coord):
        # Residual (log − background) and a 0–1 confidence along the RGT (or flat-time) axis U
        r = w["ai_low"] - bg_trace(m, w["i"], w["j"])
        f, l = w["first"], w["last"]
        r[:f], r[l + 1:] = r[f], r[l]                          # hold end values ...
        k = np.arange(NT)
        dist = np.where(w["valid"], 0, np.minimum(np.abs(k - f), np.abs(k - l))) * dt
        conf = np.where(dist < LFM_TAPER_MS, 0.5 * (1 + np.cos(np.pi * dist / LFM_TAPER_MS)), 0.0)  # ... but fade them out
        prof = well_profile(w, coord)
        return np.interp(U, prof, r), np.interp(U, prof, conf, left=0.0, right=0.0)

    def idw(x, y, w):
        return 1.0 / ((x - w["x"]) ** 2 + (y - w["y"]) ** 2 + 1.0) ** (IDW_POWER / 2)

    def residual_field(u, weights, tables):
        # Inverse-distance blend of the wells' residuals at each point's RGT value; fades to 0 where no well has data
        k = np.rint(np.clip(u, 0, 1) * (K - 1)).astype(np.int32)
        num = np.zeros(u.shape, np.float32); den = np.zeros(u.shape, np.float32); cmax = np.zeros(u.shape, np.float32)
        for W, (Rw, Cw) in zip(weights, tables):
            Wb = W[..., None] if np.ndim(W) else W
            c = Cw[k].astype(np.float32)
            num += Wb * c * Rw[k]; den += Wb * c; np.maximum(cmax, c, out=cmax)
        return num / (den + 1e-30) * cmax

    # ---- the LFM from all wells
    model = fit_background(wells)
    if model[0] == "velocity":
        print(f"\nBackground: AI ≈ {model[1]:.4g} × Vint^{model[2]:.3f}, fitted on all {len(wells)} well(s)")
        for w in wells:
            m_ = w["valid"]; r_ = np.corrcoef(w["ai_low"][m_], bg_trace(model, w["i"], w["j"])[m_])[0, 1]
            print(f"   {w['name']:12s} correlation of log and velocity background: {r_:.2f}")
    else:
        print(f"\nNo velocity model: background = AI trend with TWT ({model[2]:.3f} per ms), fitted on all wells")
    bg_label = "velocity only" if model[0] == "velocity" else "trend only"
    tables = [well_table(w, model, "rgt") for w in wells]
    bg = bg_volume(model)
    lfm_bg = lowpass(bg)
    lfm = lowpass(bg + residual_field(rgt, [idw(Xg, Yg, w) for w in wells], tables))
    del bg

    # ---- leave-one-well-out: rebuild without each well, predict it, compare with its low-passed log
    loo = {}
    if len(wells) >= 2:
        rows = []
        for h in wells:
            others = [w for w in wells if w is not h]
            m = fit_background(others)
            bg_h = bg_trace(m, h["i"], h["j"])
            preds = {bg_label: bg_h}
            for coord, label in (("time", "wells along flat time"), ("rgt", "wells along RGT")):
                tabs = [well_table(w, m, coord) for w in others]
                u = t_norm if coord == "time" else rgt[h["i"], h["j"]]
                preds[label] = bg_h + residual_field(u, [idw(Xg[h["i"], h["j"]], Yg[h["i"], h["j"]], w) for w in others], tabs)
            loo[h["name"]] = {}
            for label, p in preds.items():
                p = lowpass(p); loo[h["name"]][label] = p
                ok, y = h["valid"], h["ai_low"]
                e = (p[ok] - y[ok]) / y[ok] * 100
                rows.append(dict(well=h["name"], method=label, rms_pct=np.sqrt(np.mean(e ** 2)), bias_pct=e.mean(),
                                 correlation=np.corrcoef(p[ok], y[ok])[0, 1]))
        res = pd.DataFrame(rows)
        order = [bg_label, "wells along flat time", "wells along RGT"]
        print("\nLEAVE-ONE-WELL-OUT: each well predicted from the others (lower RMS % is better)")
        piv = res.pivot(index="well", columns="method", values="rms_pct")[order]
        piv.loc["MEAN"] = piv.mean()
        pd.options.display.float_format = "{:.2f}".format
        display(piv.rename(columns=lambda c: f"RMS % ({c})"))
        display(res.pivot(index="well", columns="method", values="correlation")[order]
                .rename(columns=lambda c: f"correlation ({c})"))
        ax = piv.drop("MEAN").plot.bar(figsize=(10, 4), color=["#999999", "#e9c46a", "#7b3fbf"], rot=0)
        ax.set_ylabel("blind-well RMS error (%)"); ax.set_title("Leave-one-well-out: lower is better"); plt.show()
    else:
        print("\n(Only one well: the leave-one-well-out test needs at least 2.)")

    # ---- plots: every well's track, a map, and sections through the first well
    n = len(wells)
    fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 6), squeeze=False)
    for ax, w in zip(axes[0], wells):
        ax.plot(w["ai"], samples, color="0.75", lw=0.6, label="log")
        ax.plot(w["ai_low"], samples, color="C3", lw=2, label=f"log ≤{LFM_CUTOFF_HZ} Hz")
        ax.plot(lfm_bg[w["i"], w["j"]], samples, color="C0", lw=1.3, ls="--", label=bg_label)
        if w["name"] in loo:
            ax.plot(loo[w["name"]]["wells along RGT"], samples, color="#7b3fbf", lw=1.5, label="blind: wells along RGT")
        ax.plot(lfm[w["i"], w["j"]], samples, color="k", lw=1, ls=":", label="final LFM")
        ax.set_ylim(samples[-1], samples[0]); ax.set_title(w["name"]); ax.set_xlabel("AI")
        for t_ in INTERVAL_MS: ax.axhline(t_, color="orange", lw=0.8, ls="--")
    axes[0][0].set_ylabel("TWT (ms)"); axes[0][0].legend(fontsize=7)
    fig.tight_layout(); plt.show()

    w0 = wells[0]
    vmin, vmax = np.nanpercentile(lfm[w0["i"]], [1, 99])
    tm = tidx(sum(INTERVAL_MS) / 2)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), gridspec_kw={"width_ratios": [1.3, 2, 2]})
    im = axes[0].imshow(lfm[:, :, tm], cmap="viridis", origin="lower", aspect="auto",
                        extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
    for w in wells:
        axes[0].plot(w["xl"], w["il"], "r*", ms=12); axes[0].annotate(w["name"], (w["xl"], w["il"]), color="w", fontsize=8,
                                                                     xytext=(4, 4), textcoords="offset points")
    axes[0].set_title(f"final LFM at {samples[tm]:.0f} ms"); axes[0].set_xlabel("crossline"); axes[0].set_ylabel("inline")
    plt.colorbar(im, ax=axes[0], label="AI")
    for ax, (title, vol) in zip(axes[1:], (("final LFM", lfm), (bg_label, lfm_bg))):
        im = ax.imshow(vol[w0["i"]].T, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax,
                       extent=(xlines[0], xlines[-1], samples[-1], samples[0]))
        for w in wells:
            if w["i"] == w0["i"] or abs(w["i"] - w0["i"]) <= 2:
                ax.axvline(w["xl"], color="red", lw=1, ls=":")
        ax.set_title(f"{title}: inline {ilines[w0['i']]} (through {w0['name']})")
        ax.set_xlabel("crossline"); ax.set_ylabel("TWT (ms)")
        for t_ in INTERVAL_MS: ax.axhline(t_, color="orange", lw=1, ls="--")
    plt.colorbar(im, ax=axes[2], label="AI")
    fig.tight_layout(); plt.show()

    OUT["lfm"] = write_like_input(lfm, f"/content/{stem}_LFM.sgy")
    OUT["lfm_background"] = write_like_input(lfm_bg, f"/content/{stem}_LFM_{bg_label.replace(' ', '_')}.sgy")

    # Decimated copy: every LFM_TRACE_STEP-th inline and crossline (first well's lines included)
    sel_i = np.arange(w0['i'] % LFM_TRACE_STEP, NI, LFM_TRACE_STEP)
    sel_x = np.arange(w0['j'] % LFM_TRACE_STEP, NX, LFM_TRACE_STEP)
    if (trace_index[np.ix_(sel_i, sel_x)] < 0).any():
        print("Some decimated grid nodes have no trace in the input: skipping the decimated file.")
    else:
        spec = segyio.spec()
        spec.iline, spec.xline = il_byte, xl_byte
        spec.ilines, spec.xlines = ilines[sel_i], xlines[sel_x]
        spec.samples, spec.sorting, spec.format = samples, 2, 5
        out_path = f"/content/{stem}_LFM_step{LFM_TRACE_STEP}.sgy"
        with segyio.open(SEGY_PATH, ignore_geometry=True) as src, segyio.create(out_path, spec) as dst:
            dst.text[0] = src.text[0]
            n = 0
            for i in sel_i:
                for j in sel_x:
                    dst.header[n] = src.header[int(trace_index[i, j])]
                    dst.trace[n] = np.ascontiguousarray(lfm[i, j], dtype=np.float32)
                    n += 1
        OUT["lfm_decimated"] = out_path
        print("wrote", out_path, f"({len(sel_i)} inlines × {len(sel_x)} crosslines, {os.path.getsize(out_path)/1e6:.1f} MB)")

## 9. Save the outputs

- `DATA_SOURCE = "drive"`: this run's outputs are copied to **<DRIVE_FOLDER>/outputs/<date_time>/**.
- `DATA_SOURCE = "upload"`: they're zipped and downloaded. If the browser blocks it, right-click the zip in the Files panel and choose **Download**.

In [ ]:
run_files = [p for p in sorted(glob.glob(f"/content/{stem}_*"))
             if not p.endswith(".zip") and os.path.getmtime(p) >= RUN_START]     # only this run's outputs
if DATA_SOURCE == "drive":
    out_dir = os.path.join(DRIVE_FOLDER, "outputs", time.strftime("%Y-%m-%d_%H%M", time.gmtime(RUN_START)))
    os.makedirs(out_dir, exist_ok=True)
    for p in run_files:
        shutil.copyfile(p, os.path.join(out_dir, os.path.basename(p))); print("  →", os.path.basename(p))
    print(f"Saved {len(run_files)} files to {out_dir}  (time in UTC)")
else:
    import zipfile
    from google.colab import files
    zip_path = f"/content/{stem}_RGT_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in run_files:
            z.write(p, os.path.basename(p)); print("  +", os.path.basename(p))
    print(f"{zip_path} ({os.path.getsize(zip_path)/1e6:.0f} MB)")
    files.download(zip_path)